# Data Center Vision-Stream Impact Classification Pipeline

Orchestrates tabular preprocessing, Earth Engine Sentinel-2 extraction, GCS upload, and optional Vertex AI training.

## Section 1: Dependencies & Environment Setup

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

import ee
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from src.eda import (
    load_building_context,
    plot_geographic_tile_footprints,
    plot_satellite_samples_gcs,
    plot_tabular_eda,
    verify_tile_alignment_gcs,
)
from src.eval import (
    display_training_summary,
    load_run_artifacts,
    plot_confusion_matrix_from_artifacts,
    plot_loss_curve_from_artifacts,
)
from src.gcs import GCS_BUCKET, GCS_FUSE_ROOT, upload_manifest
from src.preprocessing import run_tabular_preprocessing
from src.satellite import fetch_sentinel2_tensors, generate_mock_satellite_tensors

PROJECT_ID = "datacenter-summer-poc"
LOCATION = "us-central1"

ee.Initialize(project=PROJECT_ID)

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"GCP Project: {PROJECT_ID}, Region: {LOCATION}")
print(f"GCS bucket: gs://{GCS_BUCKET}/")

## Section 2: Tabular Preprocessing & Labels

In [ ]:
manifest_df = run_tabular_preprocessing("data/buildings.csv")
print(manifest_df["target_label"].value_counts().sort_index())

manifest_uri = upload_manifest("data/parsed_manifest.csv", project_id=PROJECT_ID)
print(f"Manifest available at {manifest_uri}")

## Section 2b: Tabular EDA

Explore label balance, floor-area distributions, and geographic coverage before fetching imagery.

In [ ]:
context_df = load_building_context("data/parsed_manifest.csv")
display(context_df.head(10))

plot_tabular_eda(context_df)
plt.show()

# Sample records across all three impact tiers
sample_ids = (
    context_df.groupby("target_label", group_keys=False)
    .apply(lambda group: group.head(2))
    .sort_values("target_label")
)
display(
    sample_ids[
        ["OBJECTID", "BuildingName", "latitude", "longitude", "target_label", "MaxGFA"]
    ]
)

## Section 3: Satellite Image Pipeline (Earth Engine → GCS)

Tiles upload directly to `gs://datacenter-summer-poc-data/image_tiles/` and `raw_satellite/`. Set `USE_MOCK = True` for offline dev. Use `DEV_LIMIT = 5` for a smoke test; set `None` for full extraction (~202 tiles).

In [ ]:
USE_MOCK = False
DEV_LIMIT = 5  # set to None for full extraction (~202 tiles)

if USE_MOCK:
    tile_count = generate_mock_satellite_tensors(
        manifest_path="data/parsed_manifest.csv",
        project_id=PROJECT_ID,
        limit=DEV_LIMIT,
    )
else:
    tile_count = fetch_sentinel2_tensors(
        manifest_path="data/parsed_manifest.csv",
        project_id=PROJECT_ID,
        limit=DEV_LIMIT,
    )
print(f"Uploaded {tile_count} tiles to GCS")

## Section 3b: Satellite Imagery EDA

Verify each building has aligned raw previews and processed tensors.

In [ ]:
alignment_report = verify_tile_alignment_gcs(
    manifest_path="data/parsed_manifest.csv",
    bucket=GCS_BUCKET,
    project_id=PROJECT_ID,
)
assert alignment_report["has_npy"].all(), "Some buildings are missing GCS .npy tiles"
assert alignment_report["has_raw_rgb"].all(), (
    "Some buildings are missing GCS raw RGB previews"
)
display(alignment_report.head(10))

## Section 3c: Satellite Tile Visualization

In [ ]:
qa_object_ids = (
    context_df.sort_values("MaxGFA", ascending=False)
    .groupby("target_label", group_keys=False)
    .head(1)["OBJECTID"]
    .astype(int)
    .tolist()
)
print("QA sample OBJECTIDs:", qa_object_ids)

plot_geographic_tile_footprints(context_df, sample_object_ids=qa_object_ids)
plt.show()

plot_satellite_samples_gcs(
    context_df,
    sample_object_ids=qa_object_ids,
    bucket=GCS_BUCKET,
    project_id=PROJECT_ID,
)
plt.show()

## Section 3d: GCS Storage Note

Imagery is written directly to bucket-root `image_tiles/` and `raw_satellite/` during extraction. The manifest is uploaded in Section 2.

In [ ]:
print("Training data layout:")
print(f"  gs://{GCS_BUCKET}/parsed_manifest.csv")
print(f"  gs://{GCS_BUCKET}/image_tiles/")
print(f"  gs://{GCS_BUCKET}/raw_satellite/")
print(f"Vertex fuse mount: {GCS_FUSE_ROOT}/")

## Section 4: Verify Production Modules

In [ ]:
from pathlib import Path

required_modules = [
    "src/model_def.py",
    "src/vertex_entrypoint.py",
    "src/gcs.py",
    "src/eval.py",
    "src/logging_config.py",
]
for module_path in required_modules:
    assert Path(module_path).exists(), f"Missing: {module_path}"
print("All production modules present in src/.")

## Section 5: Vertex AI Job Orchestration

Launches a Custom Training job reading from the GCS fuse mount at bucket root.

In [ ]:
from google.cloud import aiplatform

RUN_ID = datetime.utcnow().strftime("%Y%m%d-%H%M%S")


def deploy_vertex_training() -> aiplatform.CustomTrainingJob:
    container = "us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-0.py310:latest"
    aiplatform.init(project=PROJECT_ID, location=LOCATION)
    job = aiplatform.CustomTrainingJob(
        display_name=f"datacenter-vision-{RUN_ID}",
        script_path="src/vertex_entrypoint.py",
        container_uri=container,
        requirements=["torch", "torchvision", "pandas", "numpy", "matplotlib"],
    )
    job.run(
        args=[
            "--epochs",
            "10",
            "--batch-size",
            "8",
            "--training",
            GCS_FUSE_ROOT,
        ],
        replica_count=1,
        machine_type="n1-standard-4",
        accelerator_type="NVIDIA_TESLA_T4",
        accelerator_count=1,
        base_output_dir=f"gs://{GCS_BUCKET}/output/models/{RUN_ID}/",
        environment_variables={"PYTHONPATH": "src"},
    )
    return job


training_job = deploy_vertex_training()
print(f"Training job submitted. Artifacts -> gs://{GCS_BUCKET}/output/models/{RUN_ID}/")

## Section 6: Post-Training Evaluation

Download the latest run artifacts from GCS and inspect metrics and plots.

In [ ]:
artifacts_dir = load_run_artifacts(
    run_prefix=f"output/models/{RUN_ID}/",
    local_dir="artifacts",
    bucket=GCS_BUCKET,
    project_id=PROJECT_ID,
)
display_training_summary(artifacts_dir)

plot_loss_curve_from_artifacts(artifacts_dir)
plt.show()

plot_confusion_matrix_from_artifacts(artifacts_dir)
plt.show()